In [1]:
import os
import xarray as xr
import geopandas as gpd
from shapely.geometry import Point
import rioxarray as rxr

data_dir = "/storage/dlhogan/precipitation-rodeo/data/"

# Sand Castle 1: Merging SAIL datasets

In [2]:
original_file_dir = "/storage/dlhogan/precipitation-rodeo/data/raw/SAIL"
filename = "SAIL_met_all.nc"

In [ ]:
ds = xr.open_dataset(os.path.join(original_file_dir, filename))

In [ ]:
# save the merged dataset to a new NetCDF file
lat = ds["lat"].values[0]
lon = ds["lon"].values[0]
elev = ds["alt"].values[0]

# Create a GeoDataFrame and name after dataset.attrs['datastream']

gdf = gpd.GeoDataFrame(
    {
        "datastream": [ds.attrs.get("datastream", "unknown")],
        "latitude": [lat],
        "longitude": [lon],
        "elevation": [elev],
    },
    geometry=[Point(lon, lat, elev)],
    crs="EPSG:4326",
)

# Sand Castle 2: Extracting PRISM data for specific basin

In [2]:
def clip_prism(raster_path, shape_path):
    # Prepare output path
    out_path = raster_path.replace("raw", "processed").replace(".nc", "_clipped.nc")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Open raster
    with xr.open_dataset(raster_path) as ds:
        # Read shapefile
        shape = gpd.read_file(shape_path)

        # Set CRS for raster
        crs = ds.crs.attrs["crs_wkt"]
        ds = ds.rio.write_crs(crs)

        # Transform shapefile to match raster CRS
        shape = shape.to_crs(ds.rio.crs)

        # Clip raster
        clipped = ds.rio.clip(shape.geometry, shape.crs, drop=True)

        # Rename variable to ppt and set attributes
        clipped = clipped.rename({"Band1": "ppt"})
        clipped["ppt"].attrs["units"] = "mm"
        clipped["ppt"].attrs["long_name"] = "PRISM daily precipitation"

        # Write clipped file
        clipped.to_netcdf(out_path)

    # Delete original only if the clipped file was successfully written
    if os.path.exists(out_path):
        os.remove(raster_path)
        print(f"✅ Original deleted: {raster_path}")

    return

In [3]:
example_path = "/storage/dlhogan/precipitation-rodeo/data/external/PRISM/raw/prism_ppt_us_30s_20201023.nc"
shape_path = "/storage/dlhogan/precipitation-rodeo/data/geographic/East_River_lumped_HRUs_GRUs.shp"

clip_prism(example_path, shape_path)

✅ Original deleted: /storage/dlhogan/precipitation-rodeo/data/external/PRISM/raw/prism_ppt_us_30s_20201023.nc


In [3]:
ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/external/PRISM/processed/prism_ppt_us_30s_20201024_clipped.nc")

In [4]:
ds

<xarray.Dataset> Size: 8kB
Dimensions:  (lat: 44, lon: 44)
Coordinates:
  * lat      (lat) float64 352B 38.67 38.68 38.68 38.69 ... 39.01 39.02 39.03
  * lon      (lon) float64 352B -107.1 -107.1 -107.1 ... -106.8 -106.8 -106.8
Data variables:
    crs      int64 8B ...
    ppt      (lat, lon) float32 8kB ...
Attributes:
    Conventions:  CF-1.5
    GDAL:         GDAL 3.4.3, released 2022/04/22
    history:      Tue Oct 14 15:33:15 2025: GDAL CreateCopy( /nfs/pancake/u5/...

# Sand Castle 3: ERA5-Land Data

In [ ]:
import cdsapi
import pandas as pd
import xarray as xr

# Set the time zone shift as a variable so it is easy to change
TIME_ZONE_SHIFT_HOURS = -7  # UTC-7 for MDT

# Calculate the time the TIME-ZONE midnight in UTC
LOCAL_MIDNIGHT_IN_UTC = (0-TIME_ZONE_SHIFT_HOURS) % 24
TIME_STEPS = ['00:00', f"{LOCAL_MIDNIGHT_IN_UTC:02d}:00"]
client = cdsapi.Client() 
dataset = "reanalysis-era5-land"
request = {
    'product_type': ['reanalysis'],
    'variable': ['total_precipitation'],
    'date': '20240101/20240131',
    'time': TIME_STEPS,
    'area': [39.1, -107.1, 38.8, -106.8],  # North, West, South, East
    'grid': [1, 1],
    'data_format': 'netcdf',
}
result_file = client.retrieve(dataset, request).download("/storage/dlhogan/precipitation-rodeo/data/external/ERA5-Land/era5_land_20240101_20240131.zip")

2025-10-14 16:29:41,700 INFO Request ID is c4ec8237-bfd1-41c3-8c56-ed474c6ef812
2025-10-14 16:29:41,913 INFO status has been updated to accepted
2025-10-14 16:29:56,158 INFO status has been updated to successful


6e1e79d8ae6dc38756a3bf3f37402ce1.zip:   0%|          | 0.00/25.2k [00:00<?, ?B/s]

In [28]:
result_file

'/storage/dlhogan/precipitation-rodeo/data/external/ERA5-Land/era5_land_20240101_20240131.zip'

In [30]:
import zipfile
import os

# unzip if needed
if result_file.endswith(".zip"):
    with zipfile.ZipFile(result_file, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(result_file))
    # Remove the zip file after extraction
    os.remove(result_file)
    result_file = result_file.replace(".zip", ".nc")
    # rename the file to era5_land_20240101_20240131.nc
    new_file_path = os.path.join(os.path.dirname(result_file), "era5_land_20240101_20240131.nc")
    os.rename(result_file, new_file_path)
    result_file = new_file_path
ds = xr.open_dataset(
    "/storage/dlhogan/precipitation-rodeo/data/external/ERA5-Land/era5_land_20240101_20240131.nc"
)
ds

<xarray.Dataset> Size: 2kB
Dimensions:     (valid_time: 62, latitude: 1, longitude: 1)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 496B 2024-01-01 ... 2024-01-31T07...
  * latitude    (latitude) float64 8B 38.8
  * longitude   (longitude) float64 8B -107.1
    expver      (valid_time) <U4 992B ...
Data variables:
    tp          (valid_time, latitude, longitude) float32 248B ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-14T23:19 GRIB to CDM+CF via cfgrib-0.9.1...

In [14]:
# Group the data by hour
ds_grouped_by_hour = ds.groupby("valid_time.hour")

# Then create new datasets for the UTC midnight and the local midnight
i_UTC_minight, i_local_midnight = ds_grouped_by_hour.groups
ds_UTC_midnight = ds.isel(valid_time=ds_grouped_by_hour.groups[i_UTC_minight])
ds_local_midnight = ds.isel(valid_time=ds_grouped_by_hour.groups[i_local_midnight])

ds_local_midnight = ds_local_midnight.assign_coords(
    valid_time=ds_UTC_midnight.valid_time + pd.Timedelta(days=1)
)

# Subtract the UTC midnight data from the local midnight data
ds_local_to_utc_midnight = ds_UTC_midnight - ds_local_midnight
# Shift the time back one day
ds_local_to_utc_midnight = ds_local_to_utc_midnight.assign_coords(
    valid_time=ds_local_to_utc_midnight.valid_time - pd.Timedelta(days=1)
)

ds_accum_local = ds_local_midnight + ds_local_to_utc_midnight

shift = int(TIME_ZONE_SHIFT_HOURS < 0)
ds_accum_local = ds_accum_local.assign_coords(
    valid_time=ds_accum_local.valid_time + pd.Timedelta(days=shift)
)

# Sand Castle 4: Testing laser disdrometer output

In [39]:
ds = xr.open_dataset(f"{data_dir}/processed/SPLASH/SPLASH_kp_laser_disdrometer_30min.nc")

# Sand Castle 5: Working on processing AOS met data

In [5]:
import glob
import os
os.chdir("/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/")
from utils import process_sail_data
from utils.helper_funcs import convert_to_local_time
import numpy as np
from scipy import stats

files = glob.glob(f"{data_dir}/raw/SAIL/aos_mtcb/*.nc")
example_ds = xr.open_dataset(files[1])

In [6]:
precipitation_sum_vars = [v for v in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] if v in example_ds.data_vars]
precipitation_duration_vars = [v for v in process_sail_data.SAIL_PRECIPITATION_VARS['duration'] if v in example_ds.data_vars]
precipitation_rate_vars = [v for v in process_sail_data.SAIL_PRECIPITATION_VARS['rate'] if v in example_ds.data_vars]
relative_humidity_vars = [v for v in process_sail_data.SAIL_HUMIDITY_VARS['mean'] if v in example_ds.data_vars]
wind_spd_vars = [v for v in process_sail_data.SAIL_WIND_VARS['mean'] if v in example_ds.data_vars] + ["u", "v"]
wind_dir_vars = [v for v in process_sail_data.SAIL_WIND_VARS['median'] if v in example_ds.data_vars]
temperature_vars = [v for v in process_sail_data.SAIL_TEMPERATURE_VARS['mean'] if v in example_ds.data_vars]
pressure_vars = [v for v in process_sail_data.SAIL_PRESSURE_VARS['mean'] if v in example_ds.data_vars]

KeyError: 'median'

In [7]:
# calculate u and v from wind speed and direction
def calculate_wind_components(wind_speed, wind_direction):
    """
    Calculate the u and v components of wind from wind speed and direction.

    Parameters:
    wind_speed (float or np.ndarray): Wind speed in m/s.
    wind_direction (float or np.ndarray): Wind direction in degrees from north.

    Returns:
    tuple: A tuple containing the u and v components of the wind.
    """
    # Convert wind direction from degrees to radians
    wind_direction_rad = np.radians(wind_direction)

    # Calculate u and v components
    u = -wind_speed * np.sin(wind_direction_rad)  # East-West component
    v = -wind_speed * np.cos(wind_direction_rad)  # North-South component

    return u.data, v.data
def xr_mode(x, axis=None):
    """Compute the statistical mode for an xarray reduce operation."""
    mode_result = stats.mode(x, nan_policy='omit', axis=axis)
    return xr.DataArray(mode_result.mode)


u, v = calculate_wind_components(example_ds['wind_speed'].values, example_ds['wind_direction'].values)

# Add u and v to the dataset
example_ds['u'] = (('time'), u)
example_ds['v'] = (('time'), v)
example_ds['u'].attrs['units'] = 'm/s'
example_ds['v'].attrs['units'] = 'm/s'
example_ds['u'].attrs['long_name'] = 'East-West wind component'
example_ds['v'].attrs['long_name'] = 'North-South wind component'

example_ds = convert_to_local_time(example_ds, local_tz='America/Denver')

In [71]:
precipitation_sum_da      = example_ds[precipitation_sum_vars].resample(time='30min').sum()
precipitation_duration_da = example_ds[precipitation_duration_vars].resample(time='30min').sum()
precipitation_rate_da     = example_ds[precipitation_rate_vars].resample(time='30min').mean()
relative_humidity_da      = example_ds[relative_humidity_vars].resample(time='30min').mean()
wind_spd_da               = example_ds[wind_spd_vars].resample(time='30min').mean()
wind_dir_da               = example_ds[wind_dir_vars].resample(time='30min').reduce(xr_mode)
temperature_da            = example_ds[temperature_vars].resample(time='30min').mean()
pressure_da               = example_ds[pressure_vars].resample(time='30min').mean()

In [73]:
# assign units to the new data arrays
for var in precipitation_sum_vars:
    precipitation_sum_da[var].attrs['units'] = 'mm'
for var in precipitation_duration_vars:
    # convert from seconds to minutes if needed
    if precipitation_duration_da[var].attrs['units'] == 's':
        precipitation_duration_da[var] = precipitation_duration_da[var] / 60
        precipitation_duration_da[var].attrs['units'] = 'min'
for var in precipitation_rate_vars:
    precipitation_rate_da[var].attrs['units'] = 'mm/hr'
for var in relative_humidity_vars:
    relative_humidity_da[var].attrs['units'] = '%'
for var in wind_spd_vars:
    wind_spd_da[var].attrs['units'] = 'm/s'
for var in wind_dir_vars:
    wind_dir_da[var].attrs['units'] = 'deg'
for var in temperature_vars:
    temperature_da[var].attrs['units'] = 'degC'
for var in pressure_vars:
    pressure_da[var].attrs['units'] = 'hPa'

In [74]:
ds_merged = xr.merge([
        precipitation_sum_da,
        precipitation_duration_da,
        precipitation_rate_da,
        relative_humidity_da,
        wind_spd_da,
        wind_dir_da,
        temperature_da,
        pressure_da
    ])

# Sand Castle 6: Laser disdrometer processing for SAIL

In [1]:
import glob
import os
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
os.chdir(project_root)
from utils import process_sail_data
from utils.helper_funcs import convert_to_local_time
import xarray as xr
import time
import numpy as np
from scipy import stats
import pandas as pd

# Assign data directory and get files
SITE_NAME = "gothic"
data_dir = "/storage/dlhogan/precipitation-rodeo/data/"
files = glob.glob(f"{data_dir}raw/SAIL/laser_disdrometer_{SITE_NAME}/*.nc")

In [2]:
parsivel_correction_dict = { 
    'holroyd1971': [0.17, -1],
    'brandes2007': [0.178, -0.922],
    'heymsfield2004': [0.104, -0.95]
}

def correct_SAIL_parsivel_for_snow(ds, method='holroyd1971'):
    """
    Correct snowfall rate using a method discussed in Boudala et al. 2014
    """
    a = parsivel_correction_dict[method][0]
    b = parsivel_correction_dict[method][1]
    # Number density of particles
    N_D = ds['number_density_drops']
    # Fall velocity of particles summed over raw_fall_velocity
    V_D = ds['fall_velocity_calculated']
    # Class size width
    class_size_width = ds['class_size_width']

    # Apply the condition to include particle sizes from 2 to 31
    particle_size_indices = range(2, 32)
    raw_fall_velocity_indices = range(2, 32)

    # Select the relevant slices using isel
    N_D_masked = N_D.isel(particle_size=particle_size_indices)
    class_size_width_masked = class_size_width.isel(particle_size=particle_size_indices)
    V_D_masked = V_D.isel(raw_fall_velocity=raw_fall_velocity_indices)

    # Calculate the snowfall rate using vectorized operations
    result = (N_D_masked * V_D_masked * class_size_width_masked ** (3 + b)).sum(dim='particle_size').sum(dim='raw_fall_velocity')

    # Calculate the final result
    final_result = (6 * a * np.pi * 10e-4 * result)/60
    # filter to only include times with snowfall
    final_result = final_result.where(ds['weather_code'].isin([70,71,72,73,74,75,76,77,78,79,85,86,87]), ds['precip_rate'])
    return final_result

def xr_mode(x, axis=None):
    """Compute the statistical mode for an xarray reduce operation."""
    mode_result = stats.mode(x, nan_policy='omit', axis=axis)
    return xr.DataArray(mode_result.mode)

1. read data file (1-minute resolution)
2. filter to only variables we want to keep
3. filter out any bad data
4. calculate correction for snow using all methods (save these as individual arrays)
5. create accumulated variable
6. convert time to local time
7. resample to desired length using appropriate function for each variable
8. 

In [25]:
ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/processed/SAIL/laser_disdrometer_gothic/laser_disdrometer_gothic_processed_30min.nc")

# Sand Castle 7: MET station processing for SAIL

In [1]:
import glob
import os
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils import process_sail_data
from utils.helper_funcs import convert_to_local_time, qc_sail_met, calculate_wind_components
import xarray as xr
import time
import numpy as np
from scipy import stats
import pandas as pd

# Assign data directory and get files
data_dir = "/storage/dlhogan/precipitation-rodeo/data/"
files = glob.glob(f"{data_dir}raw/SAIL/met/*.nc")

vars_to_keep = [
    'atmos_pressure',
    'temp_mean',
    'rh_mean',
    'vapor_pressure_mean',
    'wspd_vec_mean',
    'wdir_vec_mean',
    'pwd_err_code',
    'pwd_mean_vis_1min',
    'pwd_precip_rate_mean_1min',
    'pwd_cumul_rain',
    'pwd_cumul_snow',
    'org_precip_rate_mean',
    'tbrg_precip_total',
    'tbrg_precip_total_corr',
    'lat',
    'lon',
    'alt',
]

In [95]:
ds = process_sail_data.initial_sail_processing(files[1], vars_to_keep=vars_to_keep)

In [54]:
# create accumulated variables
org_precip_accum= ds["org_precip_rate_mean"]/60
org_precip_accum.name = "org_precip_accum"
org_precip_accum.attrs["units"] = "mm"
org_precip_accum.attrs["long_name"] = "Original Precipitation Accumulation"

# calculate wind components
# Calculate u and v components if both wind speed and direction are present
if 'wspd_vec_mean' in ds and 'wdir_vec_mean' in ds:
    u, v = calculate_wind_components(ds['wspd_vec_mean'], ds['wdir_vec_mean'])
    ds['u'] = (('time',), u)
    ds['v'] = (('time',), v)
    ds['u'].attrs['units'] = 'm/s'
    ds['v'].attrs['units'] = 'm/s'
    ds['u'].attrs['long_name'] = 'East-West wind component'
    ds['v'].attrs['long_name'] = 'North-South wind component'

In [81]:
# resample to desired length
# accumulated variables: sum
org_precip_accum_da = org_precip_accum.resample(time='30min').sum()
pwd_cumul_rain_da = ds['pwd_cumul_rain'].resample(time='30min').sum()
pwd_cumul_snow_da = ds['pwd_cumul_snow'].resample(time='30min').sum()
tbrg_precip_total_da = ds['tbrg_precip_total'].resample(time='30min').sum()
tbrg_precip_total_corr_da = ds['tbrg_precip_total_corr'].resample(time='30min').sum()

# mean variables: mean
atmos_pressure_da = ds['atmos_pressure'].resample(time='30min').mean()
temp_mean_da = ds['temp_mean'].resample(time='30min').mean()
rh_mean_da = ds['rh_mean'].resample(time='30min').mean()
vapor_pressure_mean_da = ds['vapor_pressure_mean'].resample(time='30min').mean()
wspd_vec_mean_da = ds['wspd_vec_mean'].resample(time='30min').mean()
u_mean_da = ds['u'].resample(time='30min').mean()
v_mean_da = ds['v'].resample(time='30min').mean()
pwd_precip_rate_mean_da = ds['pwd_precip_rate_mean_1min'].resample(time='30min').mean()
org_precip_rate_mean_da = ds['org_precip_rate_mean'].resample(time='30min').mean()

# mode variables: mode
# silence small sample warning
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    wdir_vec_mean_da = ((10 * np.round(ds['wdir_vec_mean'] / 10)).astype(int)).resample(time='30min').reduce(xr_mode)
    pwd_err_code_da = (ds['pwd_err_code'].fillna(0)).resample(time='30min').reduce(xr_mode)
    pwd_mean_vis_1min_da = ds['pwd_mean_vis_1min'].resample(time='30min').reduce(xr_mode)

# first value for lat, lon, alt
lat_da = ds['lat'].resample(time='30min').first()
lon_da = ds['lon'].resample(time='30min').first()
alt_da = ds['alt'].resample(time='30min').first()

# merge all data arrays
ds_merged = xr.merge([
    org_precip_accum_da,
    pwd_cumul_rain_da,
    pwd_cumul_snow_da,
    tbrg_precip_total_da,
    tbrg_precip_total_corr_da,
    atmos_pressure_da,
    temp_mean_da,
    rh_mean_da,
    vapor_pressure_mean_da,
    wspd_vec_mean_da,
    u_mean_da,
    v_mean_da,
    pwd_precip_rate_mean_da,
    org_precip_rate_mean_da,
    wdir_vec_mean_da,
    pwd_err_code_da,
    pwd_mean_vis_1min_da,
    lat_da,
    lon_da,
    alt_da
])


# Sandbox 8: Process SAIL pluviometer data

In [2]:
import glob
import os
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils import process_sail_data
from utils.helper_funcs import convert_to_local_time
import xarray as xr
import time
import numpy as np
from scipy import stats
import pandas as pd

# Assign data directory and get files
SITE_NAME = "mtcb"
data_dir = "/storage/dlhogan/precipitation-rodeo/data/"
files = glob.glob(f"{data_dir}raw/SAIL/pluvio/*.nc")

In [6]:
vars_to_keep = [
    'intensity_rt',
    'accum_rtnrt',
    'accum_nrt',
    'accum_total_nrt',
    'maintenance_flag',
    'reset_flag',
    'intensity_rtnrt',
    'lat',
    'lon',
    'alt',
]

In [7]:
ds = process_sail_data.initial_sail_processing(files[0], vars_to_keep=vars_to_keep)

In [9]:
# fill times with 0 when maintenance_flag > 0
ds = ds.where(((ds['maintenance_flag'] == 0) | (ds['reset_flag'] == 0)), 0)

In [11]:
# resample variables to desired length
resample_length = "30min"
# accumulated variables: sum
accum_rtnrt_da = ds['accum_rtnrt'].resample(time=resample_length).sum()
accum_nrt_da = ds['accum_nrt'].resample(time=resample_length).sum()
accum_total_nrt_da = ds['accum_total_nrt'].resample(time=resample_length).sum()
# rate variables: mean
intensity_rt_da = ds['intensity_rt'].resample(time=resample_length).mean()
intensity_rtnrt_da = ds['intensity_rtnrt'].resample(time=resample_length).mean()
# first value for lat, lon, alt
lat_da = ds['lat'].resample(time=resample_length).first()
lon_da = ds['lon'].resample(time=resample_length).first()
alt_da = ds['alt'].resample(time=resample_length).first()

ds_merged = xr.merge([
    accum_rtnrt_da,
    accum_nrt_da,
    accum_total_nrt_da,
    intensity_rt_da,
    intensity_rtnrt_da,
    lat_da,
    lon_da,
    alt_da
])

In [15]:
ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/processed/SAIL/pluvio_30min.nc")

# Sandbox 9: Process SQUIRE data

Goal is to get the SQUIRE grid point from the radar over Gothic and Kettle Ponds

In [11]:
import xarray as xr
import pandas as pd
import numpy as np
import glob
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils.helper_funcs import convert_to_local_time, get_point_info
import time

def process_squire_data(files, resample_interval='30min', gothic_point_info=None, kettle_ponds_point_info=None):
    ds_list_gothic = []
    ds_list_kettle_ponds = []
    for i, file in enumerate(files[0:3]):
        print(f"Processing file {i+1}/{len(files)}: {file}")
        start = time.time()
        ds = xr.open_dataset(file)

        # convert the ds to use lat and lon as the coordinates, replacing x and y
        ds['x'] = ds['x'].assign_coords({'x': ds['lon'].values})
        ds['y'] = ds['y'].assign_coords({'y': ds['lat'].values})
        ds = ds.swap_dims({'x': 'lon', 'y': 'lat'})

        # exclude dBZ and corrected_reflectivity variables
        ds = ds.drop_vars(['DBZ', 'corrected_reflectivity', 'lowest_height'])
        # convert to local time
        ds = convert_to_local_time(ds, local_tz='America/Denver')

        # grab the nearest point to gothic
        ds_gothic = ds.sel(lon=gothic_point_info['longitude'].values
                            , lat=gothic_point_info['latitude'].values, method='nearest').squeeze()
        ds_gothic = ds_gothic.expand_dims(site=['gothic'])
        ds_kettle_ponds = ds.sel(lon=kettle_ponds_point_info['longitude'].values
                            , lat=kettle_ponds_point_info['latitude'].values, method='nearest').squeeze()
        ds_kettle_ponds = ds_kettle_ponds.expand_dims(site=['kettle_ponds'])

        # resample to resample interval
        ds_gothic_resampled = ds_gothic.resample(time=resample_interval).mean().squeeze()
        ds_kettle_ponds_resampled = ds_kettle_ponds.resample(time=resample_interval).mean().squeeze()

        # caluclate resample interval total by converting from mm/hour to mm/30min
        ds_gothic_resampled_total = ds_gothic_resampled * 0.5
        ds_kettle_ponds_resampled_total = ds_kettle_ponds_resampled * 0.5

        # rename these variables to indicate total
        for var in ds_gothic_resampled.data_vars:
            ds_gothic_resampled_total = ds_gothic_resampled_total.rename({var: var + '_total'})
        for var in ds_kettle_ponds_resampled.data_vars:
            ds_kettle_ponds_resampled_total = ds_kettle_ponds_resampled_total.rename({var: var + '_total'})

        # add unit and longname attributes for total variables
        for var in ds_gothic_resampled_total.data_vars:
            ds_gothic_resampled_total[var].attrs['units'] = 'mm'
            ds_kettle_ponds_resampled_total[var].attrs['units'] = 'mm'
            ds_gothic_resampled_total[var].attrs['long_name'] = ds_gothic_resampled[var.replace('_total', '')].attrs.get('long_name', '') + ' Total'
            ds_kettle_ponds_resampled_total[var].attrs['long_name'] = ds_kettle_ponds_resampled[var.replace('_total', '')].attrs.get('long_name', '') + ' Total'
            
        # add the total to the original dataset
        ds_gothic_resampled = ds_gothic_resampled.merge(ds_gothic_resampled_total, join='left')
        ds_kettle_ponds_resampled = ds_kettle_ponds_resampled.merge(ds_kettle_ponds_resampled_total, join='left')

        # add attributes from original dataset
        for var in ds.data_vars:
            ds_gothic_resampled[var].attrs = ds[var].attrs
            ds_kettle_ponds_resampled[var].attrs = ds[var].attrs

        # add lat, lon, elev attributes
        ds_gothic_resampled = ds_gothic_resampled.assign_attrs({
            'latitude': gothic_point_info['latitude'].astype('float32').values,
            'longitude': gothic_point_info['longitude'].astype('float32').values,
            'elevation': gothic_point_info['elevation'].astype('float32').values,
            'time_zone': 'America/Denver',  
        })
        ds_kettle_ponds_resampled = ds_kettle_ponds_resampled.assign_attrs({
            'latitude': kettle_ponds_point_info['latitude'].astype('float32').values,
            'longitude': kettle_ponds_point_info['longitude'].astype('float32').values,
            'elevation': kettle_ponds_point_info['elevation'].astype('float32').values,
            'time_zone': 'America/Denver',
        })

        # remove timezone info from time coordinate
        ds_gothic_resampled['time'] = pd.to_datetime(ds_gothic_resampled['time'].values).tz_localize(None)
        ds_kettle_ponds_resampled['time'] = pd.to_datetime(ds_kettle_ponds_resampled['time'].values).tz_localize(None)

        ds_list_gothic.append(ds_gothic_resampled)
        ds_list_kettle_ponds.append(ds_kettle_ponds_resampled)

        end = time.time()
        print(f"Finished processing file {file} in {end - start:.2f} seconds.")
    combined_ds_gothic = xr.concat(ds_list_gothic, dim='time')
    combined_ds_gothic = combined_ds_gothic.sortby('time')
    combined_ds_kettle_ponds = xr.concat(ds_list_kettle_ponds, dim='time')
    combined_ds_kettle_ponds = combined_ds_kettle_ponds.sortby('time')

    # concatenate along site dimension
    combined_ds = xr.concat([combined_ds_gothic, combined_ds_kettle_ponds], dim='site')
    return combined_ds

data_dir = "/storage/dlhogan/precipitation-rodeo/data/"
# check if SQUIRE files are present
if not os.path.exists(f"{data_dir}raw/SAIL/squire_radar/"):
    print("SQUIRE data directory not found. Download the data before proceeding.")
try:
    files = glob.glob(f"{data_dir}raw/SAIL/squire_radar/gucxprecipradarsquireS2.c1*.nc")
except Exception as e:
    print(f"Error finding SQUIRE files: {e}")

RESAMPLE_INTERVAL = '30min'
# get the lon and lat values for gothic and kettle ponds
try:
    example_gothic_ds = xr.open_dataset(f"{data_dir}processed/SAIL/met_30min.nc")
    gothic_point_info = get_point_info(example_gothic_ds)
    print('Got Gothic locations!')
except Exception as e:
    print(f"Error opening example Gothic example dataset: {e}. Make sure the file exists.")
try:
    example_kp_ds = xr.open_dataset(f"{data_dir}processed/SPLASH/asfs30_30min.nc")
    kettle_ponds_point_info = get_point_info(example_kp_ds)
    print('Got Kettle Ponds locations!')
except Exception as e:
    print(f"Error opening example Kettle Ponds example dataset: {e}. Make sure the file exists.")

combined_ds = process_squire_data(
    files, resample_interval=RESAMPLE_INTERVAL,
    gothic_point_info=gothic_point_info,
    kettle_ponds_point_info=kettle_ponds_point_info
)

Got Gothic locations!
Got Kettle Ponds locations!
Processing file 1/228: /storage/dlhogan/precipitation-rodeo/data/raw/SAIL/squire_radar/gucxprecipradarsquireS2.c1.20211201.000000.nc
Finished processing file /storage/dlhogan/precipitation-rodeo/data/raw/SAIL/squire_radar/gucxprecipradarsquireS2.c1.20211201.000000.nc in 0.25 seconds.
Processing file 2/228: /storage/dlhogan/precipitation-rodeo/data/raw/SAIL/squire_radar/gucxprecipradarsquireS2.c1.20211202.000000.nc
Finished processing file /storage/dlhogan/precipitation-rodeo/data/raw/SAIL/squire_radar/gucxprecipradarsquireS2.c1.20211202.000000.nc in 0.26 seconds.
Processing file 3/228: /storage/dlhogan/precipitation-rodeo/data/raw/SAIL/squire_radar/gucxprecipradarsquireS2.c1.20211203.000000.nc
Finished processing file /storage/dlhogan/precipitation-rodeo/data/raw/SAIL/squire_radar/gucxprecipradarsquireS2.c1.20211203.000000.nc in 0.20 seconds.


In [12]:
combined_ds

<xarray.Dataset> Size: 24kB
Dimensions:                  (site: 2, time: 144)
Coordinates:
  * site                     (site) <U12 96B 'gothic' 'kettle_ponds'
    x                        (site) float64 16B -3.75e+03 -2.25e+03
    y                        (site) float64 16B 6.5e+03 4.5e+03
    lon                      (site) float64 16B -107.0 -107.0
    lat                      (site) float64 16B 38.96 38.94
  * time                     (time) datetime64[ns] 1kB 2021-12-01 ... 2021-12...
Data variables:
    rain_rate_A              (site, time) float64 2kB nan nan ... 0.0 0.00697
    snow_rate_ws88diw        (site, time) float64 2kB nan nan nan ... nan nan
    snow_rate_m2009_1        (site, time) float64 2kB nan nan nan ... nan nan
    snow_rate_m2009_2        (site, time) float64 2kB nan nan nan ... nan nan
    snow_rate_ws2012         (site, time) float64 2kB nan nan nan ... nan nan
    rain_rate_A_total        (site, time) float64 2kB nan nan ... 0.0 0.003485
    snow_rate_ws88diw_total  (site, time) float64 2kB nan nan nan ... nan nan
    snow_rate_m2009_1_total  (site, time) float64 2kB nan nan nan ... nan nan
    snow_rate_m2009_2_total  (site, time) float64 2kB nan nan nan ... nan nan
    snow_rate_ws2012_total   (site, time) float64 2kB nan nan nan ... nan nan
Attributes: (12/27)
    command_line:          
    Conventions:           ARM-1.3 CF/Radial instrument_parameters
    process_version:       
    dod_version:           
    site_id:               
    platform_id:           
    ...                    ...
    fields:                DBZ, corrected_reflectivity, time, lowest_height, ...
    history:               
    latitude:              [38.956158]
    longitude:             [-106.987854]
    elevation:             [2886.]
    time_zone:             America/Denver

# Sandbox 10: Process SOS variables

In [209]:
import glob
os.chdir("/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/")
import utils.helper_funcs as hf
import numpy as np
storage_dir = "/storage/dlhogan/precipitation-rodeo/data/"
ds = xr.open_dataset(f"{storage_dir}processed/SOS/sos_ds_all_storage.nc")

In [210]:
# change sites array to be ['d', 'ue', 'uw', 'c']
sites = ['d', 'ue', 'uw', 'c']

wind_vars = hf.WIND_VARIABLES
wv_vars = hf.WATER_VAPOR_VARIABLES
temp_vars = hf.TEMPERATURE_VARIABLES
press_vars = hf.PRESSURE_VARIABLES
swe_vars = hf.SWE_VARIABLES
rad_vars = hf.RADIATION_VARIABLES

def fast_mode_rounded(x):
    arr = x.to_numpy()
    arr = arr[~np.isnan(arr)]  # drop NaNs fast
    if arr.size == 0:
        return np.nan
    rounded = (10 * np.round(arr / 10)).astype(int)
    vals, counts = np.unique(rounded, return_counts=True)
    return vals[np.argmax(counts)]

vars_to_keep = wind_vars + wv_vars + temp_vars + press_vars + swe_vars

# only variables at site _c
vars_to_keep = [var for var in vars_to_keep if var.endswith('_c')]

# only keep 2m, 3m, and 10m variables
vars_to_keep = [var for var in vars_to_keep if any(f"_{h}" in var for h in ['2m', '3m', '10m', 'p1', 'p2', 'p3', 'p4'])]

# remove vertical wind speed w_
vars_to_keep = [var for var in vars_to_keep if not var.startswith('w_')] + rad_vars

sub_ds = ds[vars_to_keep]
ds.close()

In [211]:
# for SWE_ variables, bfill NaN values
for var in swe_vars:
    # backward fill values
    sub_ds[var] = sub_ds[var].bfill(dim='time')
    # remove any values with large absolute differences between time steps
    swe_diff = sub_ds[var].diff(dim='time')
    large_diff_mask = np.abs(swe_diff) > 30  # threshold of 30 mm
    # set the value after the large diff to NaN
    indices_to_nan = large_diff_mask.where(large_diff_mask, drop=True).time
    sub_ds[var] = sub_ds[var].where(~sub_ds['time'].isin(indices_to_nan), np.nan)
    # bfill again
    sub_ds[var] = sub_ds[var].bfill(dim='time')
    # create a new variable called var + '_max_accum_swe'
    sub_ds[var + '_max_accum'] = (('time',), np.maximum.accumulate(sub_ds[var].values))
    sub_ds[var + '_max_accum'].attrs['units'] = sub_ds[var].attrs['units']
    sub_ds[var + '_max_accum'].attrs['long_name'] = f"Accumulated {sub_ds[var].attrs.get('long_name', var)}"

In [212]:
# convert to local time
sub_ds = hf.convert_to_local_time(sub_ds, local_tz='America/Denver')

In [ ]:
# resample to 30min
df = sub_ds.to_dataframe()
df = df.resample('30min').agg(
    {
        'SWE_p1_c': 'mean', 'SWE_p2_c': 'mean', 'SWE_p3_c': 'mean', 'SWE_p4_c': 'mean',
        'SWE_p1_c_max_accum': 'mean', 'SWE_p2_c_max_accum': 'mean', 'SWE_p3_c_max_accum': 'mean', 'SWE_p4_c_max_accum': 'mean',
        'spd_2m_c': 'mean', 'dir_2m_c': fast_mode_rounded, 'u_2m_c': 'mean', 'v_2m_c': 'mean',
        'spd_3m_c': 'mean', 'dir_3m_c': fast_mode_rounded, 'u_3m_c': 'mean', 'v_3m_c': 'mean',
        'spd_10m_c': 'mean', 'dir_10m_c': fast_mode_rounded, 'u_10m_c': 'mean', 'v_10m_c': 'mean',
        'h2o_2m_c': 'mean', 'h2o_3m_c': 'mean', 'h2o_10m_c': 'mean',
        'T_2m_c': 'mean', 'T_3m_c': 'mean', 'T_10m_c': 'mean',
        'RH_2m_c': 'mean', 'RH_3m_c': 'mean', 'RH_10m_c': 'mean',
        'P_10m_c': 'mean',
        'Rpile_out_9m_d': 'mean', 'Rpile_in_9m_d': 'mean', 'Rsw_in_9m_d': 'mean', 'Rsw_out_9m_d': 'mean',
    }
)

sub_ds_30min = df.to_xarray()

# add attributes back
for var in sub_ds_30min.data_vars:
    sub_ds_30min[var].attrs = sub_ds[var].attrs

# add global attributes
sub_ds_30min.attrs = sub_ds.attrs


<xarray.Dataset> Size: 2MB
Dimensions:             (time: 9696)
Coordinates:
  * time                (time) datetime64[ns, America/Denver] 78kB 2022-11-29...
Data variables: (12/34)
    SWE_p1_c            (time) float32 39kB 27.96 28.05 28.14 ... 0.0 0.0 0.0
    SWE_p2_c            (time) float32 39kB 21.25 21.56 21.57 ... 0.0 0.0 0.0
    SWE_p3_c            (time) float32 39kB 35.05 35.28 35.3 ... 0.0 0.0 0.0
    SWE_p4_c            (time) float32 39kB 31.32 31.44 31.71 ... 0.0 0.0 0.0
    SWE_p1_c_max_accum  (time) float32 39kB 28.22 28.3 28.31 ... 683.0 683.0
    SWE_p2_c_max_accum  (time) float32 39kB 21.29 21.62 21.79 ... 436.6 436.6
    ...                  ...
    RH_10m_c            (time) float32 39kB 63.87 48.18 48.41 ... 16.38 16.85
    P_10m_c             (time) float32 39kB 713.2 713.4 714.0 ... 720.9 720.9
    Rpile_out_9m_d      (time) float32 39kB -21.36 -31.66 -34.89 ... 7.862 nan
    Rpile_in_9m_d       (time) float32 39kB -74.59 -106.5 -104.7 ... -159.4 nan
    Rsw_in_9m_d         (time) float32 39kB -4.158 -4.515 -4.378 ... 666.0 nan
    Rsw_out_9m_d        (time) float32 39kB -0.6018 -1.114 -1.235 ... 131.7 nan
Attributes:
    project:                   SOS
    history:                   Created: 2024-03-04 08:06:41 +0000\n
    NIDAS_version:             v1.2.3
    calibration_file_path:     /h/eol/isfs/isfs/projects/SOS/ISFS/cal_files/$...
    project_config:            /h/eol/isfs/isfs/projects/SOS/ISFS/config/sos....
    wind3d_horiz_coordinates:  geographic
    file_length_seconds:       86400
    wind3d_horiz_rotation:     1
    wind3d_tilt_correction:    1

In [220]:
ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/processed/SOS/sos_ds_30min.nc")
ds

<xarray.Dataset> Size: 2MB
Dimensions:             (time: 9696)
Coordinates:
  * time                (time) datetime64[ns] 78kB 2022-11-30 ... 2023-06-19T...
Data variables: (12/34)
    SWE_p1_c            (time) float32 39kB ...
    SWE_p2_c            (time) float32 39kB ...
    SWE_p3_c            (time) float32 39kB ...
    SWE_p4_c            (time) float32 39kB ...
    SWE_p1_c_max_accum  (time) float32 39kB ...
    SWE_p2_c_max_accum  (time) float32 39kB ...
    ...                  ...
    RH_10m_c            (time) float32 39kB ...
    P_10m_c             (time) float32 39kB ...
    Rpile_out_9m_d      (time) float32 39kB ...
    Rpile_in_9m_d       (time) float32 39kB ...
    Rsw_in_9m_d         (time) float32 39kB ...
    Rsw_out_9m_d        (time) float32 39kB ...
Attributes:
    project:                   SOS
    history:                   Created: 2024-03-04 08:06:41 +0000\n
    NIDAS_version:             v1.2.3
    calibration_file_path:     /h/eol/isfs/isfs/projects/SOS/ISFS/cal_files/$...
    project_config:            /h/eol/isfs/isfs/projects/SOS/ISFS/config/sos....
    wind3d_horiz_coordinates:  geographic
    file_length_seconds:       86400
    wind3d_horiz_rotation:     1
    wind3d_tilt_correction:    1

# Sandbox 11:  Process SPLASH ASFS data from KP

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import glob
import os 
os.chdir("/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/")
import utils.helper_funcs as hf

# check if ASFS-30 files are present
if not os.path.exists(f"{data_dir}raw/SPLASH/ASFS-30_Level2_SPLASH2021-2023/"):
    print("ASFS-30 data directory not found. Download the data before proceeding.")
try:
    files = glob.glob(f"{data_dir}raw/SPLASH/ASFS-30_Level2_SPLASH2021-2023/sledseb.asfs30.level2.0.10min*.nc")
except Exception as e:
    print(f"Error finding ASFS-30 files: {e}")

In [ ]:
ds = xr.open_dataset(files[0])

vars_to_keep = [
    'lat',
    'lon',
    'altitude',
    'snow_depth',
    'atmos_pressure',
    'temp',
    'rh',
    'vapor_pressure',
    'rhi',
    'wspd_u_mean',
    'wspd_v_mean',
    'down_long_hemisp',
    'down_short_hemisp',
    'up_long_hemisp',
    'up_short_hemisp',
]
qc_vars = [v + '_qc' for v in vars_to_keep if v + '_qc' in ds.data_vars]

In [15]:
sub_ds = ds[vars_to_keep + qc_vars]

# qc relevant data when qc vars are bad
for var in vars_to_keep:
    qc_var = var + '_qc'
    if qc_var in sub_ds.data_vars:
        sub_ds[var] = sub_ds[var].where(sub_ds[qc_var] == 0, other=np.nan)

# drop qc vars
sub_ds = sub_ds.drop_vars(qc_vars)

# convert to local time
sub_ds = hf.convert_to_local_time(sub_ds, local_tz='America/Denver')

# resample to 30min
sub_ds_30min = sub_ds.resample(time='30min').mean()


In [ ]:
ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/processed/SPLASH/asfs30_30min.nc")

<xarray.Dataset> Size: 4MB
Dimensions:            (time: 31008)
Coordinates:
  * time               (time) datetime64[ns] 248kB 2021-09-28 ... 2023-07-19T...
Data variables: (12/15)
    lat                (time) float64 248kB ...
    lon                (time) float64 248kB ...
    altitude           (time) float64 248kB ...
    snow_depth         (time) float64 248kB ...
    atmos_pressure     (time) float64 248kB ...
    temp               (time) float64 248kB ...
    ...                 ...
    wspd_u_mean        (time) float64 248kB ...
    wspd_v_mean        (time) float64 248kB ...
    down_long_hemisp   (time) float64 248kB ...
    down_short_hemisp  (time) float64 248kB ...
    up_long_hemisp     (time) float64 248kB ...
    up_short_hemisp    (time) float64 248kB ...
Attributes: (12/16)
    date_created:     Fri Dec  8 10:57:03 2023
    title:            Study of Precipitation, the Lower Atmosphere and Surfac...
    institution:      NOAA Physical Sciences Laboratory (PSL) and CIRES/Unive...
    file_creator:     Michael R. Gallagher; Christopher J. Cox
    creator_email:    michael.r.gallagher@noaa.gov; christopher.j.cox@noaa.gov
    Funding:          NOAA Physical Sciences Laboratory (PSL)
    ...               ...
    conventions:      cf convention variable naming as attribute whenever pos...
    history:          processed data based on level 1 ingest files
    version:          1.0, 9/18/2023
    quality_control:  Quality control in place for the observations used in t...
    qc_flags:         -1 = No Data: Instrument was not functional and no data...
    time_zone:        America/Denver

# Sandbox 12:  Process SPLASH LPDF gauge

In [ ]:
import pandas as pd
import glob
import xarray as xr

files = glob.glob(f"{data_dir}raw/SPLASH/LPDF_gauge/*.txt")

df_list = []
for file in files:
    df = pd.read_csv(file, sep=r'\s+', header=0,)
    # select columns
    sub_df = df[['date', 'time(MST)', 'Precip_mm']]
    sub_df = sub_df.assign(
        datetime=pd.to_datetime(sub_df['date'] + ' ' + sub_df['time(MST)'], format='%Y-%m-%d %H:%M:%S')
    )

    sub_df = sub_df.set_index('datetime')
    sub_df = sub_df.drop(columns=['date', 'time(MST)'])

    # Define the full expected range
    inferred_freq = pd.infer_freq(sub_df.index)
    full_index = pd.date_range(
        start=sub_df.index.min(),
        end=sub_df.index.max(),
        freq=inferred_freq,  # fallback if not inferred
        tz=sub_df.index.tz
    )

    # Reindex to fill any gaps
    sub_df = sub_df.reindex(full_index)

    # backfill missing values
    sub_df['Precip_mm'] = sub_df['Precip_mm'].bfill()

    # drop duplicates
    sub_df = sub_df[~sub_df.index.duplicated(keep='first')]

    # drop the hour values on daylight saving time transitions in 2021, 2022, or 2023
    dst_transition_times = [
        pd.Timestamp('2021-03-14 02:00:00'),
        pd.Timestamp('2021-11-07 01:00:00'),
        pd.Timestamp('2022-03-13 02:00:00'),
        pd.Timestamp('2022-11-06 01:00:00'),
        pd.Timestamp('2023-03-12 02:00:00'),
        pd.Timestamp('2023-11-05 01:00:00'),
    ]
    sub_df = sub_df[~sub_df.index.isin(dst_transition_times)]
    
    # set time zone
    # sub_df.index = sub_df.index.tz_localize('America/Denver', nonexistent='NaT', ambiguous='NaT')

    # append to list
    df_list.append(sub_df)

# concatenate all dataframes
concat_df = pd.concat(df_list).sort_index()

# convert to xarray
ds = concat_df.to_xarray()

# rename variable to prcp
ds = ds.rename({'Precip_mm': 'prcp',
                'index':'time'})
ds['prcp'].attrs['units'] = 'mm'
ds['prcp'].attrs['long_name'] = 'LPDF Gauge Precipitation'

# add global attributes
ds.attrs['timezone'] = 'America/Denver (MST/MDT)'
ds.attrs['title'] = 'LPDF Gauge Precipitation at Kettle Ponds, CO'
ds.attrs['source'] = 'SPLASH LPDF Gauge'
ds.attrs['contact'] = 'Contact Tilden Meyers for dataset <tilden.meyers@noaa.gov>'

# save to netcdf
output_path = f"{data_dir}/processed/SPLASH/lpdf_gauge_30min.nc"

# Sandbox 13:  Process billy barr data

In [56]:
import pandas as pd
import xarray as xr
import numpy as np
df = pd.read_csv("/storage/dlhogan/precipitation-rodeo/data/raw/billy-barr/billy-barr-20211001-20230930_colsAdjusted.csv",)

/tmp/ipykernel_3441306/956999057.py:4: DtypeWarning: Columns (3,5,7,9,11,13,15,17,19,21,23,25,27,29,31,33,35,37,39,41,43,45,47,53,55,61,63,65,67,69,71,73,75,79,81,83,85) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/storage/dlhogan/precipitation-rodeo/data/raw/billy-barr/billy-barr-20211001-20230930_colsAdjusted.csv",)


In [57]:
def fast_mode_rounded(x):
    arr = x.to_numpy()
    arr = arr[~np.isnan(arr)]  # drop NaNs fast
    if arr.size == 0:
        return np.nan
    rounded = (10 * np.round(arr / 10)).astype(int)
    vals, counts = np.unique(rounded, return_counts=True)
    return vals[np.argmax(counts)]
# remove spaces from all column names
df.columns = df.columns.str.replace(' ', '')

In [58]:
vars_to_keep = [
    "date","time","windSpeed_m_per_s","flg_windSpeed","windVecMag_m_per_s","flg_windVecMag",
    "windDirec_Deg", "flg_windDirec", "mxAirTemp_Deg_C", "flg_mxAirTemp", "mnAirTemp_Deg_C", "flg_mnAirTemp",
    "avAirTemp_Deg_C", "flg_avAirTemp", "relHumidty_%", "flg_relHumidty", "baromPress_mbar", "flg_baromPress",
    "precip_mm", "flg_precip", 'accumPcpn_mm',  'flg_accumPcpn', 'pcpnS1Wire_mm',  'flg_pcpnS1Wire', 
    'PcpnS2Wire_mm', 'flg_PcpnS2Wire', 'PcpnS3Wire_mm', 'flg_PcpnS3Wire', #'SnowDepth_mm', 'flg_SnowDepth',
]
flg_vars = [var for var in vars_to_keep if var.startswith("flg_")]
non_flg_vars = [var for var in vars_to_keep if not var.startswith("flg_")]
non_flg_vars.remove("date")
non_flg_vars.remove("time")

# zip flag and non-flag variables together
for non_flg_var, flg_var in zip(non_flg_vars, flg_vars):
    # set non-flag variable to NaN where flag variable is not 0
    df[non_flg_var] = df[non_flg_var].where(df[flg_var] == 0, np.nan)

sub_df = df[vars_to_keep]
# make the index the combination of date and time
sub_df.index = pd.to_datetime(sub_df['date'] + ' ' + sub_df['time'])
sub_df = sub_df.drop(columns=['date', 'time'])

# remove spaces from all column names
sub_df.columns = sub_df.columns.str.replace(' ', '')

# replace -9999.9 with NaN in all columns except flag columns
for col in sub_df.columns:
    if col not in flg_vars:
        sub_df[col] = sub_df[col].replace(-9999.0, np.nan)

# drop flag columns
sub_df = sub_df.drop(columns=flg_vars)

# resample to 30 min intervals
sub_df_30min = sub_df.resample('30min').agg(
    {
        'windSpeed_m_per_s': 'mean',
        'windVecMag_m_per_s': 'mean',
        'windDirec_Deg': fast_mode_rounded,
        'mxAirTemp_Deg_C': 'mean',
        'mnAirTemp_Deg_C': 'mean',
        'avAirTemp_Deg_C': 'mean',
        'relHumidty_%': 'mean',
        'baromPress_mbar': 'mean',
        'precip_mm': 'sum',
        'accumPcpn_mm': 'sum',
        'pcpnS1Wire_mm': 'sum',
        'PcpnS2Wire_mm': 'sum',
        'PcpnS3Wire_mm': 'sum',
        # 'SnowDepth_mm': 'mean',
    }
)

In [61]:
# convert to xarray dataset
ds_30min = sub_df_30min.to_xarray()

# rename index to time
ds_30min = ds_30min.rename({'index': 'time'})

# rename variables by removing units after '_' from name, then save units in attrs
var_rename_dict = {
    'windSpeed_m_per_s': 'windSpeed',
    'windVecMag_m_per_s': 'windVecMag',
    'windDirec_Deg': 'windDirec',
    'mxAirTemp_Deg_C': 'mxAirTemp',
    'mnAirTemp_Deg_C': 'mnAirTemp',
    'avAirTemp_Deg_C': 'avAirTemp',
    'relHumidty_%': 'relHumidty',
    'baromPress_mbar': 'baromPress',
    'precip_mm': 'precip',
    'accumPcpn_mm': 'accumPcpn',
    'pcpnS1Wire_mm': 'pcpnS1Wire',
    'PcpnS2Wire_mm': 'pcpnS2Wire',
    'PcpnS3Wire_mm': 'pcpnS3Wire',
    # 'SnowDepth_mm': 'SnowDepth',
}
for old_name, new_name in var_rename_dict.items():
    ds_30min = ds_30min.rename({old_name: new_name})
    # save units in attrs
    if old_name.endswith('_m_per_s'):
        ds_30min[new_name].attrs['units'] = 'm/s'
    elif old_name.endswith('_Deg'):
        ds_30min[new_name].attrs['units'] = 'degrees'
    elif old_name.endswith('_Deg_C'):
        ds_30min[new_name].attrs['units'] = 'degC'
    elif old_name.endswith('_mbar'):
        ds_30min[new_name].attrs['units'] = 'mbar'
    elif old_name.endswith('_mm'):
        ds_30min[new_name].attrs['units'] = 'mm'
    elif old_name.endswith('_%'):
        ds_30min[new_name].attrs['units'] = '%'

In [65]:
# add lat, lon, alt variables
ds_30min['lat'] = 38.963
ds_30min['lat'].attrs['units'] = 'degrees_north'

ds_30min['lon'] = -106.993
ds_30min['lon'].attrs['units'] = 'degrees_east'

ds_30min['alt'] = 2917.6
ds_30min['alt'].attrs['units'] = 'meters'

# add global attributes
ds_30min.attrs['site_name'] = 'billy_barr'
ds_30min.attrs['location'] = 'East River Valley, Gothic, Colorado, USA'
ds_30min.attrs['data_source'] = 'Data provided by the billy barr meteorological station, data obtained from https://wrcc.dri.edu/cgi-bin/rawMAIN.pl?corbil'
ds_30min.attrs['processing_notes'] = 'Quality control flags were used to remove erroneous data. Data were resampled to 30 minute intervals using mean for continuous ' \
                                        'variables and sum for precipitation variables. Wind direction was resampled using mode after rounding to nearest 10 degrees.'
ds_30min.attrs['date_created'] = pd.Timestamp.now().isoformat()

# Sand Castle 15: Process PRISM data
Goal is to get the PRISM 800-m grid point over Gothic/Kettle Ponds

In [21]:
import xarray as xr
import pandas as pd
import numpy as np
import glob
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils.helper_funcs import convert_to_local_time, get_point_info

# get the lon and lat values for gothic and kettle ponds
example_gothic_ds = xr.open_dataset(f"{data_dir}processed/SAIL/met_30min.nc")
example_kp_ds = xr.open_dataset(f"{data_dir}processed/SPLASH/asfs30_30min.nc")
gothic_point_info = get_point_info(example_gothic_ds)
kettle_ponds_point_info = get_point_info(example_kp_ds)

# check if PRISM files are present
if not os.path.exists(f"{data_dir}external/PRISM/processed/"):
    print("PRISM data directory not found. Download the data before proceeding.")
try:
    files = glob.glob(f"{data_dir}external/PRISM/processed/prism_ppt_us_30s*.nc")
except Exception as e:
    print(f"Error finding PRISM files: {e}")

In [22]:
ds = xr.open_dataset(files[0])

# select the nearest point to gothic
ds_gothic = ds.sel(lon=gothic_point_info['longitude'].values
                    , lat=gothic_point_info['latitude'].values, method='nearest').squeeze()
ds_gothic = ds_gothic.expand_dims(site=['gothic'])
ds_kettle_ponds = ds.sel(lon=kettle_ponds_point_info['longitude'].values
                    , lat=kettle_ponds_point_info['latitude'].values, method='nearest').squeeze()
ds_kettle_ponds = ds_kettle_ponds.expand_dims(site=['kettle_ponds'])

In [58]:
# extract the date from the filenames
for file in files:
    date_str = file.split('_')[-2]
    date = pd.to_datetime(date_str, format='%Y%m%d')
    ds = xr.open_dataset(file)
    # add a time dimension
    ds = ds.expand_dims(time=[date])
    # select the nearest point to gothic
    ds_gothic = ds.sel(lon=gothic_point_info['longitude'].values
                        , lat=gothic_point_info['latitude'].values, method='nearest').squeeze()
    ds_gothic = ds_gothic.expand_dims(site=['gothic'])
    ds_kettle_ponds = ds.sel(lon=kettle_ponds_point_info['longitude'].values
                        , lat=kettle_ponds_point_info['latitude'].values, method='nearest').squeeze()
    ds_kettle_ponds = ds_kettle_ponds.expand_dims(site=['kettle_ponds'])

    # concatenate to single dataset
    ds_sites = xr.concat([ds_gothic, ds_kettle_ponds], dim='site')
    if 'ds_all' in locals():
        ds_all = xr.concat([ds_all, ds_sites], dim='time')
    else:
        ds_all = ds_sites
    

In [ ]:
xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/processed/PRISM/prism_site_data.nc").sel(site='kettle_ponds')['ppt'].plot()

NameError: name 'xr' is not defined

# Sand Castle 16: Process ERA5-Land data
Goal is to get the ERA5-Land 9-km grid point over Gothic/Kettle Ponds

In [83]:
import xarray as xr
import pandas as pd
import numpy as np
import glob
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils.helper_funcs import convert_to_local_time, get_point_info

# check if ERA5-Land files are present
if not os.path.exists(f"{data_dir}external/ERA5-Land/"):
    print("ERA5-Land data directory not found. Download the data before proceeding.")
try:
    files = glob.glob(f"{data_dir}external/ERA5-Land/era5_land_20211001_20230930.nc")
except Exception as e:
    print(f"Error finding ERA5-Land file: {e}")

In [ ]:
era5_land_ds = xr.open_dataset(files[0])
# get the lon and lat values for gothic and kettle ponds
example_gothic_ds = xr.open_dataset(f"{data_dir}processed/SAIL/met_30min.nc")
example_kp_ds = xr.open_dataset(f"{data_dir}processed/SPLASH/asfs30_30min.nc")
gothic_point_info = get_point_info(example_gothic_ds)
kettle_ponds_point_info = get_point_info(example_kp_ds)

In [ ]:
era5_land_point = era5_land_ds.sel(longitude=gothic_point_info['longitude'].values, latitude=gothic_point_info['latitude'].values, method='nearest').squeeze()

# change to local time
era5_land_point = convert_to_local_time(era5_land_point, local_tz='America/Denver', time_variable='valid_time')

# convert from m to mm for precipitation variables
era5_land_point['tp'] = era5_land_point['tp'] * 1000
era5_land_point['tp'].attrs['units'] = 'mm'
# add longname
era5_land_point['tp'].attrs['long_name'] = 'Total Precipitation'
# add timezone attribute
era5_land_point.attrs['timezone'] = 'America/Denver (MST/MDT)'

# rename valid_time to time
era5_land_point = era5_land_point.swap_dims({'valid_time': 'time'})

# remove timezone
era5_land_point['time'] = pd.to_datetime(era5_land_point['time'].values).tz_localize(None)

# drop valid_time, number, exper 
era5_land_point = era5_land_point.drop_vars(['valid_time', 'number', 'expver'])

# save to netcdf
output_path = f"{data_dir}/processed/ERA5-Land/era5_land_gothic_1hr.nc"

In [106]:
ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data//processed/ERA5-Land/era5_land_gothic_1hr.nc")

# Sand Castle 17: Optical rain gauge

In [2]:
import glob
import os
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils import process_sail_data
from utils.helper_funcs import convert_to_local_time, qc_sail_met, calculate_wind_components
import xarray as xr
import time
import numpy as np
from scipy import stats
import pandas as pd

# Assign data directory and get files
data_dir = "/storage/dlhogan/precipitation-rodeo/data/"
files = glob.glob(f"{data_dir}raw/SAIL/maws/*.nc")

In [6]:
xr.open_dataset("/storage/dlhogan/synoptic_sublimation/sail_data/winter_21_22/met_20211001_20220930.nc")

<xarray.Dataset> Size: 134MB
Dimensions:                       (time: 524933, bound: 2)
Coordinates:
  * time                          (time) datetime64[ns] 4MB 2021-10-01 ... 20...
Dimensions without coordinates: bound
Data variables: (12/51)
    base_time                     (time) datetime64[ns] 4MB ...
    time_offset                   (time) datetime64[ns] 4MB ...
    time_bounds                   (time, bound) datetime64[ns] 8MB ...
    atmos_pressure                (time) float32 2MB ...
    qc_atmos_pressure             (time) int32 2MB ...
    temp_mean                     (time) float32 2MB ...
    ...                            ...
    qc_logger_volt                (time) int32 2MB ...
    logger_temp                   (time) float32 2MB ...
    qc_logger_temp                (time) int32 2MB ...
    lat                           (time) float32 2MB ...
    lon                           (time) float32 2MB ...
    alt                           (time) float32 2MB ...
Attributes: (12/21)
    command_line:                met_ingest -s guc -f M1 -RD --max-runtime 0
    Conventions:                 ARM-1.3
    process_version:             ingest-met-4.53-0.el7
    dod_version:                 met-b1-11.2
    input_source:                /data/reproc/D221103.2/collection/guc/gucmet...
    site_id:                     guc
    ...                          ...
    averaging_interval_comment:  The time assigned to each data point indicat...
    org:                         Optical Rain Gauge
    tbrg:                        Tipping Bucket Rain Gauge
    pwd:                         Present Weather Detector
    tbrg_precip_corr_info:       0.000000 * tbrg_precip_total^2 + 0.000000 * ...
    history:                     created by user yahmad on machine prod-proc5...